####Notebook de Entrada

Notebook de entrada principal do algoritmo de pulmão, os dados sao extraidos das seguintes tabelas:<br>
<br>
`gold_corporativo_ia.corporativo.tb_gold_mov_exame`<br>
`gold_corporativo_ia.corporativo.tb_gold_mov_paciente` 
<br>
<br>Após realizar o filtro dos exames os dados sao enriquecidos com as informações dos pacientes, posterior a vw resultante é usada para inserção dos dados na tabela final.

####Variáveis de Ambiente

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")

In [0]:
environment = dbutils.widgets.get("environment")

In [0]:
if environment in ["hml", "prd"]:
    environment_tbl = ""
else:
    environment_tbl = f"{environment}_"

In [0]:
OUTPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_pulmao"
VW_BALIZADORA = f"{environment_tbl}vw_balizador"
CATALOG = "ia"

In [0]:
print('Environment:   ', environment_tbl)
print('Tabela destino:', OUTPUT_TABLENAME)
print('VW balizadora: ', VW_BALIZADORA)
print('VW balizadora: ', CATALOG)

In [0]:
# spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{OUTPUT_TABLENAME}""")
# spark.sql(f"""DELETE FROM {CATALOG}.{OUTPUT_TABLENAME} WHERE DATE(dataExecucaoModelo) = current_date()""")


In [0]:
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
# WITH base AS (
#   SELECT
#       e.id_unidade                                     AS idunidade,
#       e.nme_hospital                                   AS unidade,
#       p.id_paciente                                    AS id_pct,
#       p.cli_nome                                       AS paciente,
#       p.cli_genero                                     AS sexo,
#       p.cli_data_nascimento                            AS nascimento,
#       p.doc_cpf                                        AS cpf,
#       p.cli_telefone_numero                            AS telefonePaciente,
#       p.cli_telefone_ddd                               AS telefonePacienteDDD,
#       e.tp_procedimento                                AS procedimento,
#       e.num_pedido_integracao                          AS num_pedido_integracao,
#       e.nme_regional_hospital                          AS nme_regional_hospital,
#       e.nme_convenio                                   AS nme_convenio,

#       regexp_replace(lower(coalesce(e.dsc_procedimento_integracao, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

#       /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
#       FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
#       CASE
#         WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo|ressonancia|ress|rm)'
#           THEN 'CT'
#         ELSE 'IND'
#       END                                              AS modalidade,
#       translate(
#         regexp_replace(
#           lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS exame,

#       e.dt_dia_exame                                   AS dataexame,
#       translate(
#         regexp_replace(
#           lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS tipoexame,

#       CAST(e.id_exame AS STRING)                       AS an,

#       e.dt_liberacao_laudo                             AS DataLaudoLiberado,
#       array_join(
#         transform(e.proced_lista_exames, x -> x.laudo_transformado),
#         '\n'
#       ) AS Laudo,
#       COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
#       CASE
#         WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
#         WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
#         ELSE 0
#       END AS idstatus

#   FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
#   LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
#          ON p.id_paciente = e.id_patient
#   WHERE DATE(e.dt_liberacao_laudo) >= date_sub(current_date(), 30)
# ),
# filtrada AS (
#   SELECT *
#   FROM base
#   WHERE
#     UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
#     AND (
#         exame_nr LIKE '%torax%' OR exame_nr LIKE '%tórax%'
#       OR exame_nr LIKE '%coronar%' OR exame_nr LIKE '%calcio%' OR exame_nr LIKE '%cálcio%'
#       OR exame_nr LIKE '%abdom%'
#       OR (
#           exame_nr LIKE '%angio%'
#           AND (
#                 exame_nr LIKE '%torax%' OR exame_nr LIKE '%tórax%'
#             OR  exame_nr LIKE '%pulmonar%'
#             OR  exame_nr LIKE '%aorta torac%'
#             OR  exame_nr LIKE '%toracoabdom%'
#             OR  exame_nr LIKE '%pulm(ã|a)o%'
#             OR  exame_nr LIKE '%pulm(õ|o)es%'
#             OR  exame_nr LIKE '%cervical%'
#             OR  exame_nr LIKE '%abd%'
#           )
#           AND exame_nr NOT LIKE '%cran%'
#         )
#         )
#     AND (
#       LOWER(COALESCE(Laudo,'')) RLIKE 'n[óo]dulo?s?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'nodular(?:es)?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'micron[óo]dulo?s?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'noduliforme'
#       OR LOWER(COALESCE(Laudo,'')) LIKE '%n&oacute;dul%'        
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'n&oacute;dulo?s?'     
#     )
#     AND DataLaudoFinal IS NOT NULL
#     AND unidade IN("HOSPITAL COPA DOR", "CLINICA SAO VICENTE", "HOSPITAL E MATERNIDADE SAO LUIZ ANALIA FRANCO", "HOSPITAL SAO LUIZ MORUMBI",
#                    "HOSPITAL E MATERNIDADE SAO LUIZ UNIDADE ITAIM", "HOSPITAL QUINTA DOR", "HOSPITAL CAXIAS DOR", "HOSPITAL SAO LUIZ JABAQUARA",
#                    "HOSPITAL RIO BARRA", "HOSPITAL BARRA DOR", "HOSPITAL RIOS DOR", "HOSPITAL ASSUNCAO", "HOSPITAL VILLA LOBOS",
#                    "HOSPITAL E MATERNIDADE SAO LUIZ SAO CAETANO", "HOSPITAL GLORIA DOR", "HOSPITAL BRASIL MAUA", "HOSPITAL BRASIL SANTO ANDRE",
#                    "HOSPITAL SANTA ISABEL", "HOSPITAL SAO LUIZ OSASCO", "HOSPITAL E MATERNIDADE RIBEIRAO PIRES", "HOSPITAL BARTIRA",
#                    "HOSPITAL OESTE DOR", "HOSPITAL CENTRAL LESTE", "HOSPITAL NITEROI DOR", "HOSPITAL SANTA CRUZ", "HOSPITAL BRASIL SANTO ANDRE",
#                    "HOSPITAL CENTRAL DO TATUAPE (AVICCENA)", "HOSPITAL SAO LUIZ GUARULHOS", "HOSPITAL SAO LUIZ ALPHAVILLE", "HOSPITAL CENTRAL SUL",
#                    "CENTRO DE IMAGEM COPA D'OR", "COPA D'OR - DAY CLINIC SOROCABA", "HOSPITAL CAXIAS D'OR", "HOSPITAL COPA D'OR",
#                    "HOSPITAL GLORIA D'OR", "HOSPITAL NITEROI D'OR", "HOSPITAL NORTE D'OR", "HOSPITAL OESTE D'OR", "HOSPITAL QUINTA D'OR",
#                    "HOSPITAL RIOS D'OR", "HOSPITAL SAO LUIZ ITAIM", "HOSPITAL SAO LUIZ SAO BERNARDO", "HOSPITAL SAO LUCAS",
#                    "HOSPITAL BARRA D'OR I AYRTON SENNA")
# ),

# dedup AS (
#   SELECT
#     f.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY an
#       ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#     ) AS rn,
#     CASE WHEN ROW_NUMBER() OVER (
#              PARTITION BY an
#              ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#          ) = 1
#          THEN 1 ELSE 0 END AS an_duplicado
#   FROM filtrada f
# )
# SELECT
#   idunidade,
#   unidade,
#   id_pct,
#   paciente,
#   telefonePacienteDDD,
#   telefonePaciente,
#   sexo,
#   nascimento,
#   idade_anos,
#   cpf,
#   modalidade,
#   exame,
#   num_pedido_integracao,
#   nme_regional_hospital,
#   nme_convenio,
#   dataexame,
#   tipoexame,
#   an,
#   an_duplicado,
#   idstatus,
#   -- 4 AS idstatus,                 -- mantido
#   -- idLaudo,                       
#   DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
#   Laudo,
#   -- current_timestamp() as datCarga
#   CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
# FROM dedup
# where idade_anos between 20 and 85
# """)

In [0]:
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
# WITH base AS (
#   SELECT
#       e.id_unidade                                     AS idunidade,
#       e.nme_hospital                                   AS unidade,
#       p.id_paciente                                    AS id_pct,
#       p.cli_nome                                       AS paciente,
#       p.cli_genero                                     AS sexo,
#       p.cli_data_nascimento                            AS nascimento,
#       p.doc_cpf                                        AS cpf,
#       p.cli_telefone_numero                            AS telefonePaciente,
#       p.cli_telefone_ddd                               AS telefonePacienteDDD,
#       e.tp_procedimento                                AS procedimento,
#       e.num_pedido_integracao                          AS num_pedido_integracao,
#       e.nme_regional_hospital                          AS nme_regional_hospital,
#       e.nme_convenio                                   AS nme_convenio,

#       regexp_replace(lower(coalesce(e.dsc_procedimento_integracao, cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') as exame_nr,

#       /*calculo o numero de meses entre as duas datas e divido por 12 para ter o valor em anos*/
#       FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,
#       CASE
#         WHEN exame_nr RLIKE '(tomografia|angio|tc|tomo|ressonancia|ress|rm)'
#           THEN 'CT'
#         ELSE 'IND'
#       END                                              AS modalidade,
#       translate(
#         regexp_replace(
#           lower(coalesce(exame_nr, cast(e.cod_procedimento as string), '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS exame,

#       e.dt_dia_exame                                   AS dataexame,
#       translate(
#         regexp_replace(
#           lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
#           '[\\u00A0\\u2007\\u202F]', ' '
#         ),
#         'áàãâäéèêëíìîïóòõôöúùûüç',
#         'aaaaaeeeeiiiiooooouuuuc'
#       ) AS tipoexame,

#       CAST(e.id_exame AS STRING)                       AS an,

#       e.dt_liberacao_laudo                             AS DataLaudoLiberado,
#       array_join(
#         transform(e.proced_lista_exames, x -> x.laudo_transformado),
#         '\n'
#       ) AS Laudo,
#       COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,
#       CASE
#         WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
#         WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
#         ELSE 0
#       END AS idstatus

#   FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame   e
#   LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
#          ON p.id_paciente = e.id_patient
#   WHERE DATE(e.dt_liberacao_laudo) >= date_sub(current_date(), 30)
# ),
# filtrada AS (
#   SELECT *
#   FROM base
#   WHERE
#     UPPER(trim(procedimento)) IN  ('IMG', 'IMA')
#     AND (
#       -- 1) RM de tórax
#       (
#         (LOWER(exame_nr) LIKE '%rm%'
#         OR LOWER(exame_nr) LIKE '%ressonancia%'
#         OR LOWER(exame_nr) LIKE '%ressonância%'
#         OR LOWER(exame_nr) LIKE '%resso%')
#         AND (LOWER(exame_nr) LIKE '%torax%' OR LOWER(exame_nr) LIKE '%tórax%')
#       )

#       OR

#       -- 2) Tomografia (TC) com cervical/torax/abdome
#       (
#         (LOWER(exame_nr) LIKE '%tc%'
#         OR LOWER(exame_nr) LIKE '%tomo%'
#         OR LOWER(exame_nr) LIKE '%tomografia%')
#         AND (
#           LOWER(exame_nr) LIKE '%cervical%'
#           OR LOWER(exame_nr) LIKE '%torax%' OR LOWER(exame_nr) LIKE '%tórax%'
#           OR LOWER(exame_nr) LIKE '%abd%' OR LOWER(exame_nr) LIKE '%abdom%'
#           OR LOWER(exame_nr) LIKE '%calcio%' OR LOWER(exame_nr) LIKE '%cálcio%'
#           OR LOWER(exame_nr) LIKE '%coronar%'
#         )
#       )

#       OR

#       -- 3) Angiotomografia com cervical/torax/abdome (e não cran)
#       (
#         LOWER(exame_nr) LIKE '%angio%'
#         AND LOWER(exame_nr) NOT LIKE '%colangio%'
#         AND (
#           LOWER(exame_nr) LIKE '%cervical%'
#           OR LOWER(exame_nr) LIKE '%torax%' OR LOWER(exame_nr) LIKE '%tórax%'
#           OR LOWER(exame_nr) LIKE '%abd%' OR LOWER(exame_nr) LIKE '%abdom%'
#           OR LOWER(exame_nr) LIKE '%pulmonar%'
#           OR LOWER(exame_nr) LIKE '%aorta torac%'
#           OR LOWER(exame_nr) LIKE '%toracoabdom%'
#           OR LOWER(exame_nr) LIKE '%calcio%' OR LOWER(exame_nr) LIKE '%cálcio%'
#           OR LOWER(exame_nr) LIKE '%coronar%'
#         )
#         AND LOWER(exame_nr) NOT LIKE '%cran%'
#       )
#     )
#     AND (
#       LOWER(COALESCE(Laudo,'')) RLIKE 'n[óo]dulo?s?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'nodular(?:es)?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'micron[óo]dulo?s?'
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'noduliforme'
#       OR LOWER(COALESCE(Laudo,'')) LIKE '%n&oacute;dul%'        
#       OR LOWER(COALESCE(Laudo,'')) RLIKE 'n&oacute;dulo?s?'     
#     )
#     AND DataLaudoFinal IS NOT NULL
#     AND idunidade IN(1,3,4,5,6,11,13,15,16,20,22,27,36,38,39,42,54,56,59,62,66,67,69,70,71,74,80,81,84,86,88,97,98,100,101,103,106,108,109,111,129,133,134,135,141,147,149,154,156,162,191,199,203,210,215,218,220,223,226,228,231,236,257,276,286,293,373,396,402,414,520,533,553,555,573,593,653,714,715,733,793,833,834,873,874,893,993,1013,1053,1054,1055,1074,1093,1133,1153,1173,1193,1213,1217,1235,1237,1239,1240,1241,1244,1248,1260,1276,1295,1297,1321,1322,1323,1324,1325,1329,1333,1334,1336,1341,1353,1373,1413,1415,1433,1453,1473,1493,1513,1514,1515,1534,1535,1536,1537,1538,1553,1573,1574,1595,1596,1633,1654,1655,1657,1693,1713,100950,100965,100968,100970,100974,100976,100999,101027,101043,200001,200002,200013,200019,200020,200021,200022,200023,200029,200033,200034,200035,200036,200038,200041,200043,200086,200091,200092,200095,200097,200101,200104,200114,200121,200128,200129,200130,200131,200135,200139,200140,200146,2,10,12,14,17,19,23,25,28,30,32,33,34,35,41,43,46,47,53,55,60,64,65,68,76,78,83,85,87,90,91,92,94,95,102,105,107,110,112,115,117,120,123,125,136,152,159,160,161,176,185,187,190,206,211,237,246,252,258,259,267,280,282,285,288,353,393,395,399,400,401,413,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,473,474,475,476,513,514,515,516,521,534,535,554,713,753,794,813,933,953,973,1073,1214,1216,1218,1233,1246,1250,1252,1253,1255,1257,1258,1261,1273,1274,1275,1277,1313,1318,1319,1327,1328,1335,1337,1338,1342,1343,1434,1455,1593,1594,1613,1615,1660,1661,1662,100943,100946,100981,100990,100994,101000,101001,101002,101004,101014,101031,101032,101033,200000,200003,200017,200028,200030,200031,200032,200039,200040,200042,200045,200046,200047,200079,200081,200083,200084,200099,200107,200109,200110,200117,200118,200122,200125,200126,200149,200153)
# ),

# dedup AS (
#   SELECT
#     f.*,
#     ROW_NUMBER() OVER (
#       PARTITION BY an
#       ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#     ) AS rn,
#     CASE WHEN ROW_NUMBER() OVER (
#              PARTITION BY an
#              ORDER BY TIMESTAMP(DataLaudoFinal) DESC
#          ) = 1
#          THEN 1 ELSE 0 END AS an_duplicado
#   FROM filtrada f
# )
# SELECT
#   idunidade,
#   unidade,
#   id_pct,
#   paciente,
#   telefonePacienteDDD,
#   telefonePaciente,
#   sexo,
#   nascimento,
#   idade_anos,
#   cpf,
#   modalidade,
#   exame,
#   num_pedido_integracao,
#   nme_regional_hospital,
#   nme_convenio,
#   dataexame,
#   tipoexame,
#   an,
#   an_duplicado,
#   idstatus,
#   -- 4 AS idstatus,                 -- mantido
#   -- idLaudo,                       
#   DataLaudoFinal AS DataLaudoLiberado,  -- >>> usa a data escolhida
#   Laudo,
#   -- current_timestamp() as datCarga
#   CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
# FROM dedup
# where idade_anos between 20 and 85
# """)

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW {VW_BALIZADORA} AS
WITH base AS (
  SELECT
      e.id_unidade                                     AS idunidade,
      e.nme_hospital                                   AS unidade,
      p.id_paciente                                    AS id_pct,
      p.cli_nome                                       AS paciente,
      p.cli_genero                                     AS sexo,
      p.cli_data_nascimento                            AS nascimento,
      p.doc_cpf                                        AS cpf,
      p.cli_telefone_numero                            AS telefonePaciente,
      p.cli_telefone_ddd                               AS telefonePacienteDDD,
      e.tp_procedimento                                AS procedimento,
      e.num_pedido_integracao                          AS num_pedido_integracao,
      e.nme_regional_hospital                          AS nme_regional_hospital,
      e.nme_convenio                                   AS nme_convenio,

      regexp_replace(
        lower(coalesce(e.proced_descricao_ajustado, cast(e.cod_procedimento as string), '')),
        '[\\u00A0\\u2007\\u202F]',
        ' '
      ) AS exame_nr,

      regexp_replace(lower(coalesce(e.dsc_codigo, '')), '[\\u00A0\\u2007\\u202F]', ' ') AS dsc_codigo_txt,
      regexp_replace(lower(coalesce(cast(e.cod_procedimento as string), '')), '[\\u00A0\\u2007\\u202F]', ' ') AS cod_procedimento_txt,

      FLOOR(months_between(DATE(e.dt_dia_exame), DATE(p.cli_data_nascimento)) / 12) AS idade_anos,

      CASE
        WHEN regexp_replace(
               lower(coalesce(e.dsc_procedimento_integracao, cast(e.cod_procedimento as string), '')),
               '[\\u00A0\\u2007\\u202F]',
               ' '
             ) RLIKE '(tomografia|angio|tc|tomo|ressonancia|ress|rm)'
          THEN 'CT'
        ELSE 'IND'
      END AS modalidade,

      translate(
        regexp_replace(
          lower(coalesce(
            regexp_replace(
              lower(coalesce(e.dsc_procedimento_integracao, cast(e.cod_procedimento as string), '')),
              '[\\u00A0\\u2007\\u202F]',
              ' '
            ),
            cast(e.cod_procedimento as string),
            ''
          )),
          '[\\u00A0\\u2007\\u202F]',
          ' '
        ),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS exame,

      e.dt_dia_exame                                   AS dataexame,

      translate(
        regexp_replace(
          lower(coalesce(e.nme_origem_exame, e.tp_proced_classe_encontro, '')),
          '[\\u00A0\\u2007\\u202F]',
          ' '
        ),
        'áàãâäéèêëíìîïóòõôöúùûüç',
        'aaaaaeeeeiiiiooooouuuuc'
      ) AS tipoexame,

      CAST(e.id_exame AS STRING)                       AS an,
      e.dt_liberacao_laudo                             AS DataLaudoLiberado,

      array_join(
        transform(e.proced_lista_exames, x -> x.laudo_transformado),
        '\n'
      ) AS Laudo,

      COALESCE(e.dt_liberacao_laudo, e.dt_previsao_laudo ) AS DataLaudoFinal,

      CASE
        WHEN e.dt_liberacao_laudo IS NOT NULL THEN 5
        WHEN e.dt_previsao_laudo  IS NOT NULL THEN 4
        ELSE 0
      END AS idstatus

  FROM gold_corporativo_ia.corporativo.tb_gold_mov_exame e
  LEFT JOIN gold_corporativo_ia.corporativo.tb_gold_mov_paciente p
         ON p.id_paciente = e.id_patient
  WHERE DATE(e.dt_liberacao_laudo) >= date_sub(current_date(), 30)
),

filtrada AS (
  SELECT *
  FROM base
  WHERE
    UPPER(trim(procedimento)) IN ('IMG', 'IMA')
    AND (
      -- 1) RM de tórax
      (
        (
          exame_nr LIKE '%rm%'
          OR exame_nr LIKE '%ressonancia%'
          OR exame_nr LIKE '%ressonância%'
          OR exame_nr LIKE '%resso%'
          OR exame_nr LIKE '%rnm%'
          OR dsc_codigo_txt LIKE '%rm%'
          OR dsc_codigo_txt LIKE '%ressonancia%'
          OR dsc_codigo_txt LIKE '%ressonância%'
          OR dsc_codigo_txt LIKE '%resso%'
          OR dsc_codigo_txt LIKE '%rnm%'
          OR cod_procedimento_txt LIKE '%rm%'
          OR cod_procedimento_txt LIKE '%ressonancia%'
          OR cod_procedimento_txt LIKE '%ressonância%'
          OR cod_procedimento_txt LIKE '%resso%'
          OR cod_procedimento_txt LIKE '%rnm%'
        )
        AND
        (
          exame_nr LIKE '%torax%'
          OR exame_nr LIKE '%tórax%'
          OR dsc_codigo_txt LIKE '%torax%'
          OR dsc_codigo_txt LIKE '%tórax%'
          OR cod_procedimento_txt LIKE '%torax%'
          OR cod_procedimento_txt LIKE '%tórax%'
        )
      )

      OR

      -- 2) Tomografia (TC) com cervical/torax/abdome
      (
        (
          exame_nr LIKE '%tc%'
          OR exame_nr LIKE '%tomo%'
          OR exame_nr LIKE '%tomografia%'
          OR dsc_codigo_txt LIKE '%tc%'
          OR dsc_codigo_txt LIKE '%tomo%'
          OR dsc_codigo_txt LIKE '%tomografia%'
          OR cod_procedimento_txt LIKE '%tc%'
          OR cod_procedimento_txt LIKE '%tomo%'
          OR cod_procedimento_txt LIKE '%tomografia%'
        )
        AND
        (
          exame_nr LIKE '%cervical%'
          OR exame_nr LIKE '%torax%'
          OR exame_nr LIKE '%tórax%'
          OR exame_nr LIKE '%abd%'
          OR exame_nr LIKE '%abdom%'
          OR exame_nr LIKE '%calcio%'
          OR exame_nr LIKE '%cálcio%'
          OR exame_nr LIKE '%coronar%'

          OR dsc_codigo_txt LIKE '%cervical%'
          OR dsc_codigo_txt LIKE '%torax%'
          OR dsc_codigo_txt LIKE '%tórax%'
          OR dsc_codigo_txt LIKE '%abd%'
          OR dsc_codigo_txt LIKE '%abdom%'
          OR dsc_codigo_txt LIKE '%calcio%'
          OR dsc_codigo_txt LIKE '%cálcio%'
          OR dsc_codigo_txt LIKE '%coronar%'

          OR cod_procedimento_txt LIKE '%cervical%'
          OR cod_procedimento_txt LIKE '%torax%'
          OR cod_procedimento_txt LIKE '%tórax%'
          OR cod_procedimento_txt LIKE '%abd%'
          OR cod_procedimento_txt LIKE '%abdom%'
          OR cod_procedimento_txt LIKE '%calcio%'
          OR cod_procedimento_txt LIKE '%cálcio%'
          OR cod_procedimento_txt LIKE '%coronar%'
        )
      )

      OR

      -- 3) Angiotomografia com cervical/torax/abdome (e não cran)
      (
        (
          exame_nr LIKE '%angio%'
          OR dsc_codigo_txt LIKE '%angio%'
          OR cod_procedimento_txt LIKE '%angio%'
        )
        AND NOT (
          exame_nr LIKE '%colangio%'
          OR dsc_codigo_txt LIKE '%colangio%'
          OR cod_procedimento_txt LIKE '%colangio%'
        )
        AND (
          exame_nr LIKE '%cervical%'
          OR exame_nr LIKE '%torax%'
          OR exame_nr LIKE '%tórax%'
          OR exame_nr LIKE '%abd%'
          OR exame_nr LIKE '%abdom%'
          OR exame_nr LIKE '%pulmonar%'
          OR exame_nr LIKE '%aorta torac%'
          OR exame_nr LIKE '%toracoabdom%'
          OR exame_nr LIKE '%calcio%'
          OR exame_nr LIKE '%cálcio%'
          OR exame_nr LIKE '%coronar%'

          OR dsc_codigo_txt LIKE '%cervical%'
          OR dsc_codigo_txt LIKE '%torax%'
          OR dsc_codigo_txt LIKE '%tórax%'
          OR dsc_codigo_txt LIKE '%abd%'
          OR dsc_codigo_txt LIKE '%abdom%'
          OR dsc_codigo_txt LIKE '%pulmonar%'
          OR dsc_codigo_txt LIKE '%aorta torac%'
          OR dsc_codigo_txt LIKE '%toracoabdom%'
          OR dsc_codigo_txt LIKE '%calcio%'
          OR dsc_codigo_txt LIKE '%cálcio%'
          OR dsc_codigo_txt LIKE '%coronar%'

          OR cod_procedimento_txt LIKE '%cervical%'
          OR cod_procedimento_txt LIKE '%torax%'
          OR cod_procedimento_txt LIKE '%tórax%'
          OR cod_procedimento_txt LIKE '%abd%'
          OR cod_procedimento_txt LIKE '%abdom%'
          OR cod_procedimento_txt LIKE '%pulmonar%'
          OR cod_procedimento_txt LIKE '%aorta torac%'
          OR cod_procedimento_txt LIKE '%toracoabdom%'
          OR cod_procedimento_txt LIKE '%calcio%'
          OR cod_procedimento_txt LIKE '%cálcio%'
          OR cod_procedimento_txt LIKE '%coronar%'
        )
        AND NOT (
          exame_nr LIKE '%cran%'
          OR dsc_codigo_txt LIKE '%cran%'
          OR cod_procedimento_txt LIKE '%cran%'
        )
      )
    )
    AND (
      LOWER(COALESCE(Laudo,'')) RLIKE 'n[óo]dulo?s?'
      OR LOWER(COALESCE(Laudo,'')) RLIKE 'nodular(?:es)?'
      OR LOWER(COALESCE(Laudo,'')) RLIKE 'micron[óo]dulo?s?'
      OR LOWER(COALESCE(Laudo,'')) RLIKE 'noduliforme'
      OR LOWER(COALESCE(Laudo,'')) LIKE '%n&oacute;dul%'
      OR LOWER(COALESCE(Laudo,'')) RLIKE 'n&oacute;dulo?s?'
    )
    AND DataLaudoFinal IS NOT NULL
    AND idunidade IN(1,3,4,5,6,11,13,15,16,20,22,27,36,38,39,42,54,56,59,62,66,67,69,70,71,74,80,81,84,86,88,97,98,100,101,103,106,108,109,111,129,133,134,135,141,147,149,154,156,162,191,199,203,210,215,218,220,223,226,228,231,236,257,276,286,293,373,396,402,414,520,533,553,555,573,593,653,714,715,733,793,833,834,873,874,893,993,1013,1053,1054,1055,1074,1093,1133,1153,1173,1193,1213,1217,1235,1237,1239,1240,1241,1244,1248,1260,1276,1295,1297,1321,1322,1323,1324,1325,1329,1333,1334,1336,1341,1353,1373,1413,1415,1433,1453,1473,1493,1513,1514,1515,1534,1535,1536,1537,1538,1553,1573,1574,1595,1596,1633,1654,1655,1657,1693,1713,100950,100965,100968,100970,100974,100976,100999,101027,101043,200001,200002,200013,200019,200020,200021,200022,200023,200029,200033,200034,200035,200036,200038,200041,200043,200086,200091,200092,200095,200097,200101,200104,200114,200121,200128,200129,200130,200131,200135,200139,200140,200146,2,10,12,14,17,19,23,25,28,30,32,33,34,35,41,43,46,47,53,55,60,64,65,68,76,78,83,85,87,90,91,92,94,95,102,105,107,110,112,115,117,120,123,125,136,152,159,160,161,176,185,187,190,206,211,237,246,252,258,259,267,280,282,285,288,353,393,395,399,400,401,413,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,473,474,475,476,513,514,515,516,521,534,535,554,713,753,794,813,933,953,973,1073,1214,1216,1218,1233,1246,1250,1252,1253,1255,1257,1258,1261,1273,1274,1275,1277,1313,1318,1319,1327,1328,1335,1337,1338,1342,1343,1434,1455,1593,1594,1613,1615,1660,1661,1662,100943,100946,100981,100990,100994,101000,101001,101002,101004,101014,101031,101032,101033,200000,200003,200017,200028,200030,200031,200032,200039,200040,200042,200045,200046,200047,200079,200081,200083,200084,200099,200107,200109,200110,200117,200118,200122,200125,200126,200149,200153
    )
),

dedup AS (
  SELECT
    f.*,
    ROW_NUMBER() OVER (
      PARTITION BY an
      ORDER BY TIMESTAMP(DataLaudoFinal) DESC
    ) AS rn,
    CASE
      WHEN ROW_NUMBER() OVER (
        PARTITION BY an
        ORDER BY TIMESTAMP(DataLaudoFinal) DESC
      ) = 1 THEN 1 ELSE 0
    END AS an_duplicado
  FROM filtrada f
)

SELECT
  idunidade,
  unidade,
  id_pct,
  paciente,
  telefonePacienteDDD,
  telefonePaciente,
  sexo,
  nascimento,
  idade_anos,
  cpf,
  modalidade,
  exame,
  num_pedido_integracao,
  nme_regional_hospital,
  nme_convenio,
  dataexame,
  tipoexame,
  an,
  an_duplicado,
  idstatus,
  DataLaudoFinal AS DataLaudoLiberado,
  Laudo,
  CAST(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo') AS TIMESTAMP_NTZ) AS dataExecucaoModelo
FROM dedup
WHERE idade_anos BETWEEN 20 AND 85
""")

In [0]:
spark.sql(f"""CREATE TABLE IF NOT EXISTS {CATALOG}.{OUTPUT_TABLENAME}
            USING DELTA
            AS
            SELECT *
            FROM {VW_BALIZADORA}
            WHERE 1 = 0
            """)

In [0]:
# spark.sql(f"""
# ALTER TABLE ia.{OUTPUT_TABLENAME}
# ADD COLUMNS (
#   num_pedido_integracao STRING,
#   nme_regional_hospital STRING,
#   nme_convenio STRING)
# """)

In [0]:
spark.sql(f"""
INSERT INTO {CATALOG}.{OUTPUT_TABLENAME} (
    idunidade,
    unidade,
    id_pct,
    paciente,
    telefonePacienteDDD,
    telefonePaciente,
    sexo,
    nascimento,
    idade_anos,
    cpf,
    modalidade,
    exame,
    num_pedido_integracao,
    nme_regional_hospital,
    nme_convenio,
    dataexame,
    tipoexame,
    an,
    an_duplicado,
    idstatus,
    DataLaudoLiberado,
    Laudo,
    dataExecucaoModelo
)
SELECT
    s.idunidade,
    s.unidade,
    s.id_pct,
    s.paciente,
    s.telefonePacienteDDD,
    s.telefonePaciente,
    s.sexo,
    s.nascimento,
    s.idade_anos,
    s.cpf,
    s.modalidade,
    s.exame,
    s.num_pedido_integracao,
    s.nme_regional_hospital,
    s.nme_convenio,
    s.dataexame,              
    s.tipoexame,
    s.an,
    s.an_duplicado,
    s.idstatus,
    s.DataLaudoLiberado,
    s.Laudo,
    s.dataExecucaoModelo
FROM {VW_BALIZADORA} s
LEFT ANTI JOIN ia.{OUTPUT_TABLENAME} t
    ON t.an        = s.an
""")

In [0]:
spark.sql(f"""
SELECT
  CAST(dataExecucaoModelo AS DATE) AS dataExecucao,
  COUNT(*) AS total_exames
FROM {CATALOG}.{OUTPUT_TABLENAME}
GROUP BY CAST(dataExecucaoModelo AS DATE)
ORDER BY dataExecucao DESC
""").display()

In [0]:
dbutils.notebook.exit("OK")

In [0]:
# spark.sql(f"""
# SELECT *
# FROM {CATALOG}.{OUTPUT_TABLENAME}
# WHERE cast(dataExecucaoModelo as date) = current_date()
# """).display()

In [0]:
# spark.sql(f"""
# SELECT
#   CAST(DataLaudoLiberado AS DATE) AS data_exame,
#   unidade,
#   COUNT(*) AS total_exames
# FROM {CATALOG}.{OUTPUT_TABLENAME}
# GROUP BY
#   CAST(DataLaudoLiberado AS DATE),
#   unidade
# ORDER BY
#   data_exame DESC,
#   unidade
# """).display()